# Pracuś.pl

## Overview

Pobranie danych w postaci ofert pracy z portalu pracuj.pl i przygotowanie ich przekrojowej analizy w Power BI.

### Pozyskanie danych

Wykorzystuję [aktora apify](https://console.apify.com/actors/uyfjMbPtsvmRBL4EL) i pobieram 250 ogłoszeń z ostatnich 30 dni dla frazy kluczowej “data engineer”. Wynik otrzymuję w formacie JSON.

` file_url = 'dataset_pracuj_2026-10-15.json'`

### Załadowanie danych

Dane zostają załadowane do **ramki pandas** jak również (opcjonalnie) do lokalnej instancji **bazy danych**.

In [ ]:
import pandas as pd
import sqlalchemy
import dotenv
import os
import json

dotenv.load_dotenv()

username = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')
port = os.getenv('DB_PORT', '1433') 


file_url = 'dataset_pracuj_2026-10-15.json'
connection = sqlalchemy.create_engine(f'mssql+pyodbc://{username}:{password}@{server}:{port}/{database}?driver=ODBC+Driver+17+for+SQL+Server')

dataframe = pd.read_json(file_url)

# Tworzenie kopii oryginalnego DataFrame na potrzeby późniejszej analizy lub porównań
original_df = dataframe.copy()

# Konwersja kolumn złożonych (słowniki/listy) do ciągów JSON
for col in dataframe.columns:
    if dataframe[col].apply(lambda x: isinstance(x, (dict, list))).any():
        dataframe[col] = dataframe[col].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)

dataframe.to_sql('pracuj_pl', con=connection, if_exists='replace', index=False)


## Python

### Chmura tagów z umiejętnościami

Wykreślimy pierwszy wykres zawierający najczęściej wymagane technologie dla tego stanowiska - bez transformacji i czyszczenia danych aby zobaczyć szybko pierwsze rezultaty.

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def generate_wordcloud(df, column_name):
    # Spłaszczanie listy technologii do jednej listy
    all_technologies = []
    for tech_list in df[column_name].dropna():
        if isinstance(tech_list, list):
            # Zamiana znaków spacji na podkreślenia w nazwach technologii
            all_technologies.extend([tech.replace(" ", "_") for tech in tech_list])
            
    # Generowanie chmury słów
    text = " ".join(all_technologies)
    if not text:
        print(f"Brakuje danych w kolumnie '{column_name}' do wygenerowania chmury słów.")
        return
        
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    
    # Pokazanie chmury słów
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Najczęściej wymagane technologie')
    plt.show()

# wygenerowanie chmury słów dla kolumny 'technologiesExpected'
print(original_df['technologiesExpected'].head())  # Wyświetlenie pierwszych kilku wierszy kolumny 'technologiesExpected'
generate_wordcloud(original_df, 'technologiesExpected')

### Przykładowy wynik

![Przykładowy wynik](./img/wordcloud.png)

<img src="./img/wordcloud.png">

## SQL

### Ekstrakcja danych

Wybieramy podzbióry interesujących nas danych do dalszej analizy oraz "normalizujemy" je do użycia w modelu danych Power BI. Wiele danych znajduje się w zagnieżdżonych listach i strukturach JSON dlatego je wyodrębnimy do oddzielnych tabel i połączymy ID oferty.

In [ ]:
SELECT
-- tabela faktów
[id] as offer_id,
[title],
JSON_VALUE([company], '$.name') AS name,
JSON_VALUE([company], '$.profileUrl') AS company_profile_url,
JSON_VALUE([company], '$.logoUrl') AS company_logo_url
FROM [datafreak].[dbo].[pracuj_pl]

SELECT
-- wynagrodzenie
[id] as offer_id,
salary.contractType,
salary.[from] as min_salary,
salary.[to] as max_salary,
salary.currency,
salary.timeUnit,
salary.kind
FROM [datafreak].[dbo].[pracuj_pl]
CROSS APPLY OPENJSON([salaries])
  WITH (
    contractType NVARCHAR(50) '$.contractType',
    [from] INT '$.from',
    [to] INT '$.to',
    currency NVARCHAR(10) '$.currency',
    timeUnit NVARCHAR(20) '$.timeUnit',
    kind NVARCHAR(50) '$.kind'
) salary

SELECT
-- miasta
[id] as offer_id,
m.[value] as miasta
FROM [datafreak].[dbo].[pracuj_pl]
CROSS APPLY OPENJSON([locations]) as m

SELECT
-- tryby pracy
[id] as offer_id,
t.[value] as tryb_pracy
FROM [datafreak].[dbo].[pracuj_pl]
CROSS APPLY OPENJSON([workModes]) as t

SELECT
-- poziom stanowiska
[id] as offer_id,
p.[value] as poziom_stanowiska
FROM [datafreak].[dbo].[pracuj_pl]
CROSS APPLY OPENJSON([positionLevels]) as p

SELECT
-- branże
[id] as offer_id,
cat.[name] as rodzaj,
cat.[parent] as branza
FROM [datafreak].[dbo].[pracuj_pl]
CROSS APPLY OPENJSON([categories])
  WITH (
    [name] VARCHAR(255) '$.name',
    [parent] VARCHAR(255) '$.parent'
) cat

SELECT
-- wymagane technologie
[id] as offer_id,
req.[value] as tech_wymagane,
opt.[value] as tech_opcjonalne
FROM [datafreak].[dbo].[pracuj_pl]
CROSS APPLY OPENJSON([technologiesExpected]) as req
CROSS APPLY OPENJSON(technologiesOptional) as opt

SELECT
-- typ umowy
[id] as offer_id,
t.[value] as typ_kontraktu
FROM [datafreak].[dbo].[pracuj_pl]
CROSS APPLY OPENJSON([typesOfContract]) as t

SELECT
-- dodatkowe informacje uzupełniające
    t.id,
    j.type,
    e.[value] AS element
FROM [datafreak].[dbo].[pracuj_pl] t
CROSS APPLY OPENJSON(t.[descriptionSections]) 
WITH (
    type NVARCHAR(50) '$.type',
    elements NVARCHAR(MAX) '$.elements' AS JSON
) j
CROSS APPLY OPENJSON(j.elements) e
WHERE j.type IN ('responsibilities', 'requirements-expected', 'benefits')

## Power BI

### Model danych

Model danych typu "gwiazda". Zwracam uwagę na filtrowanie każdej tabeli z tabelą faktów w obu kierunkach.

![Model](img/model_pbi.png)

### Gotowy dashboard

Wykorzystując zgromadzone dane możemy zbudować dashboard jak poniżej

![PBI](img/dashboard_pbi.png)


## Zapytajmy AI

Dodajmy do naszej analizy lokalny LLM i zbudujmy prosty "czat z danymi".

### Łączymy się z danymi

Do połączenia z lokalną bazą MS SQL wykorzytuję gotowe MCP jako bibliotękę Python skonfigurowane i użyte później w LM Studio z pobranym modelem. Zatem tech-stack wygląda następująco:

1. Python i zainstalowana biblioteka za pomocą PIP **mssql_mcp_server**
2. LM Studio z pobranym modelem - w tym przypadku **qwen3-8b**
3. Skonfigurowane MCP w LM Studio z podanymi danymi lokalnej bazy danych - przykład poniżej

In [ ]:
    "mssql-local": {
      "command": "C:\\MCPServers\\mssql-env\\Scripts\\python.exe",
      "args": [
        "-m",
        "mssql_mcp_server.server"
      ],
      "env": {
        "MSSQL_HOST": "127.0.0.1,14330",
        "MSSQL_DRIVER": "ODBC Driver 18 for SQL Server",
        "MSSQL_DATABASE": "XYZ",
        "MSSQL_USER": "XYZ",
        "MSSQL_PASSWORD": "XYZ",
        "TrustServerCertificate": "yes"
      }
    }



### Wynik rozmowy z LLM

Pomijając wydajność na domowym komputerze z powodzeniem można używać języka naturalnego do "rozmowy z danymi". Przykładowy wynik poniżej

#### Prompt
> Work with table dbo.pracuj_pl. What technical skills are mostly required on the positions in the database?

#### Wynik

The most required technical skills for positions in the **dbo.pracuj_pl** table are:

🔥 Top 10 Most Common Technical Skills
1. **Python** (Extremely common across all roles)
2. **SQL** (Core skill for data jobs)
3. **AWS** (Cloud infrastructure dominance)
4. **Microsoft Azure** (Strong presence in Polish market)
5. **Google Cloud Platform (GCP)** 
6. **Docker** (Containerization standard)
7. **Kubernetes** (Orchestration platform)
8. **Git** (Version control essential)
9. **Spark/PySpark** (Big data processing)
10. **Java** (Enterprise application development)

🚀 Emerging Trends
- **LLM/GenAI**: Growing demand for AI/ML skills
- **Cloud Native**: Kubernetes, Terraform, CI/CD pipelines
- **Data Engineering**: Snowflake, Databricks, Delta Lake
- **ETL/ELT**: dbt, Apache Airflow

📊 Notable Patterns
- 68% of roles require **Python or SQL**
- 45% mention **cloud platforms** (AWS/Azure/GCP)
- **Data engineering** roles dominate with Spark/Databricks
- **DevOps/Cloud Infrastructure** skills are highly valued

Would you like me to show the full list of all mentioned technologies or analyze specific skill categories?

## Odświeżanie danych w przyszłości

Jeśli w przyszłości chciałbym odświeżyć dane i/ lub wykorzystać większą próbę danych - proces jest następujący:

1. Uruchomienie "aktora" Apify z podaniem próbki danych (ilości ofert) oraz słowa kluczowego (np. stanowiska)
2. Eksport danych do pliku JSON
3. Uruchomienie skryptu ładjącego dane do (lokalnej) bazy MS SQL
4. Odświeżenie raporów w Power BI i obublikowanie w Power BI Public lub...
5. ... zapytanie asystenta AI 


&copy; cezary@datafreak.pl